# 📚 숙제 — 수학 풀이 모드 구현

**목표**: `lab.ipynb`에서 만든 계산기 에이전트에 수학 풀이 모드를 추가하세요.

수학 풀이 모드는 단순 계산이 아니라 **풀이 과정을 단계별로 설명**하면서 계산 도구를 활용합니다.

### 평가 기준

| 항목 | 배점 |
|------|------|
| State 설계 (`problem`, `solution_steps` 추가) | 20점 |
| Phoenix `math_solver_system` 프롬프트 Pull 사용 | 20점 |
| 계산기 도구(arithmetic/calculus/matrix_calc) 활용 | 30점 |
| Phoenix Trace 존재 (실행 기록 확인 가능) | 15점 |
| 테스트 케이스 3개 통과 | 15점 |

### 제출 방법
Phoenix experiment URL을 제출하세요. (코드 파일은 선택 제출)
→ Phoenix UI → Experiments 탭 → 실험 클릭 → URL 복사

---
## §0 환경 설정

`pre_setup.ipynb`를 완료했다면 아래 셀을 실행하세요.

In [ ]:
# ── 프록시 설정 로드 ──────────────────────────────────────────────────────────
# 프록시 ON/OFF: .env 파일에서 USE_PROXY=true/false 로 제어합니다.
from proxy_config import make_llm, make_eval_model, proxy_patched_anthropic

In [ ]:
import os
from phoenix.otel import register
from openinference.instrumentation.langchain import LangChainInstrumentor
from phoenix.client import Client

# Phoenix Tracing 연결
tracer_provider = register(
    project_name="math-agent-homework",
    endpoint="http://localhost:6006/v1/traces",
)
LangChainInstrumentor().instrument(tracer_provider=tracer_provider)

client = Client()
print("✅ 환경 설정 완료")

---
## §1 기존 코드 재사용

`graph.py`에서 이미 구현된 것들을 import합니다.

In [ ]:
# graph.py에서 기존 구현 재사용
from graph import (
    tools,           # arithmetic, calculus, matrix_calc
    llm,             # ChatAnthropic
    llm_with_tools,  # bind_tools 된 LLM
    tool_executor,   # ToolNode
    mode_router_v2,  # LLM 기반 모드 분류기 (이번엔 3-way로 확장)
    pull_prompt,     # Phoenix Prompt Hub에서 프롬프트 가져오기
)

print("재사용 가능한 컴포넌트:")
print(f"  tools: {[t.name for t in tools]}")
print(f"  llm: {llm.model}")
print(f"  tool_executor: {type(tool_executor).__name__}")

---
## §2 State 확장

기존 `AgentState`에 수학 풀이 전용 필드를 추가하세요.

In [ ]:
from typing import Annotated, Literal, TypedDict
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages

class MathAgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    mode: str
    # ── TODO: 아래 두 필드를 추가하세요 ──────────────────────
    # problem: str          # 사용자가 제시한 수학 문제 원문
    # solution_steps: list  # 단계별 풀이 과정 (각 단계를 str로 저장)
    # ─────────────────────────────────────────────────────────

# 힌트: TypedDict에 필드를 추가하면 됩니다.
# 예: problem: str

print("MathAgentState 정의 완료")
print("TODO: problem, solution_steps 필드를 추가하세요")

from pydantic import BaseModel
from langchain_core.messages import SystemMessage, HumanMessage

class ModeDecision3(BaseModel):
    mode: Literal["chat", "calculator", "math_solver"]
    reason: str

# TODO: 아래 _CLASSIFIER_PROMPT_3WAY를 수정해 math_solver 모드를 추가하세요
_CLASSIFIER_PROMPT_3WAY = """사용자 메시지를 분류하세요.

calculator: ???  (기존 정의 유지)

math_solver: ???  ← 이 부분을 채우세요
  예시: ???

chat: ???"""

_classifier_3way = make_llm(model="claude-haiku-4-5-20251001", temperature=0) \
    .with_structured_output(ModeDecision3)

def mode_router_3way(state: MathAgentState) -> Literal["chat", "calculator", "math_solver"]:
    last_msg = state["messages"][-1]
    result = _classifier_3way.invoke([
        SystemMessage(content=_CLASSIFIER_PROMPT_3WAY),
        HumanMessage(content=last_msg.content),
    ])
    print(f"  [Router] mode={result.mode}, reason={result.reason}")
    return result.mode

# 테스트
test_cases = [
    "23 * 47 계산해줘",           # → calculator
    "이 미분방정식을 풀어줘: ...",  # → math_solver
    "미적분이 뭔가요?",             # → chat
]
print("TODO: 분류기 프롬프트를 완성한 후 아래를 실행하세요")
# for q in test_cases:
#     fake_state = {"messages": [HumanMessage(content=q)], "mode": "", "problem": "", "solution_steps": []}
#     mode_router_3way(fake_state)

In [ ]:
from pydantic import BaseModel
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import SystemMessage, HumanMessage

class ModeDecision3(BaseModel):
    mode: Literal["chat", "calculator", "math_solver"]
    reason: str

# TODO: 아래 _CLASSIFIER_PROMPT_3WAY를 수정해 math_solver 모드를 추가하세요
_CLASSIFIER_PROMPT_3WAY = """사용자 메시지를 분류하세요.

calculator: ??? (기존 정의 유지)

math_solver: ???  ← 이 부분을 채우세요
  예시: ???

chat: ???"""

_classifier_3way = ChatAnthropic(model="claude-haiku-4-5-20251001", temperature=0) \
    .with_structured_output(ModeDecision3)

def mode_router_3way(state: MathAgentState) -> Literal["chat", "calculator", "math_solver"]:
    last_msg = state["messages"][-1]
    result = _classifier_3way.invoke([
        SystemMessage(content=_CLASSIFIER_PROMPT_3WAY),
        HumanMessage(content=last_msg.content),
    ])
    print(f"  [Router] mode={result.mode}, reason={result.reason}")
    return result.mode

# 테스트
test_cases = [
    "23 * 47 계산해줘",           # → calculator
    "이 미분방정식을 풀어줘: ...",  # → math_solver
    "미적분이 뭔가요?",             # → chat
]
print("TODO: 분류기 프롬프트를 완성한 후 아래를 실행하세요")
# for q in test_cases:
#     fake_state = {"messages": [HumanMessage(content=q)], "mode": "", "problem": "", "solution_steps": []}
#     mode_router_3way(fake_state)

---
## §4 math_solver_node 구현

핵심 구현 부분입니다.

In [ ]:
from langchain_core.messages import SystemMessage

def math_solver_node(state: MathAgentState) -> dict:
    """
    수학 풀이 노드
    
    동작:
    1. Phoenix에서 'math_solver_system' 프롬프트 Pull
    2. 사용자 문제를 분석하고 단계별 풀이 계획 수립
    3. 계산이 필요한 부분은 도구(arithmetic/calculus/matrix_calc)를 사용
    4. 풀이 과정을 solution_steps에 기록
    5. 최종 답 반환
    
    힌트:
    - pull_prompt("math_solver_system")으로 프롬프트를 가져오세요
    - llm_with_tools를 사용하면 필요할 때 Tool을 자동 호출합니다
    - state["problem"]에 원문 문제를 저장하세요
    """
    # TODO: 아래를 구현하세요
    # 1. pull_prompt("math_solver_system") 호출
    # 2. SystemMessage + state["messages"]로 messages 구성
    # 3. llm_with_tools.invoke(messages) 호출
    # 4. state 업데이트 반환

    # 임시 구현 (교체 필요)
    system_prompt = "TODO: pull_prompt()로 교체하세요"
    return {
        "messages": [],
        "mode": "math_solver",
        # "problem": ???,
        # "solution_steps": ???,
    }

print("math_solver_node 뼈대 완성")
print("TODO: 위 함수를 구현하세요")

---
## §5 전체 그래프 조립

3-way 라우터와 math_solver_node를 전체 그래프에 통합하세요.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode

# 기존 노드 재사용
from graph import chat_node, calculator_node, should_continue

# TODO: MathAgentState를 사용하는 전체 그래프를 조립하세요
builder = StateGraph(MathAgentState)

# 노드 등록
builder.add_node("chat_node", chat_node)
builder.add_node("calculator_node", calculator_node)
builder.add_node("math_solver_node", math_solver_node)  # 새로 추가
builder.add_node("tool_executor", ToolNode(tools))

# TODO: 엣지를 연결하세요
# 힌트:
# builder.add_conditional_edges(START, mode_router_3way, {...})
# math_solver_node도 tool call 루프가 필요합니다
# (math_solver_node → should_continue → tool_executor → math_solver_node)

# builder.add_edge(...)
# ...

# math_graph = builder.compile()
print("TODO: 엣지를 연결하고 그래프를 컴파일하세요")

---
## §6 테스트 케이스 3개

아래 문제들을 구현한 에이전트로 풀어보세요.

In [ ]:
# 테스트 케이스 1: 미분 문제 (풀이 과정 + 계산)
PROBLEM_1 = """다음 함수를 미분하고 극값을 구해주세요.
f(x) = x³ - 6x² + 9x + 1
풀이 과정을 단계별로 설명해주세요."""

# TODO: math_graph가 구현되면 아래 주석을 해제하세요
# result1 = math_graph.invoke({
#     "messages": [HumanMessage(content=PROBLEM_1)],
#     "mode": "",
#     "problem": PROBLEM_1,
#     "solution_steps": [],
# })
# print("테스트 1 결과:")
# print(result1["messages"][-1].content)
# print("\n풀이 단계:")
# for i, step in enumerate(result1.get("solution_steps", []), 1):
#     print(f"  {i}. {step}")

print("문제 1:", PROBLEM_1[:80], "...")

In [ ]:
# 테스트 케이스 2: 행렬 문제
PROBLEM_2 = """다음 연립방정식을 행렬 방법으로 풀어주세요.
2x + y = 5
x + 3y = 10

행렬 Ax = b 형태로 변환하고, 역행렬을 이용해 풀이하세요."""

# TODO: math_graph로 테스트하세요
print("문제 2:", PROBLEM_2[:80], "...")

In [ ]:
# 테스트 케이스 3: 적분 응용 문제
PROBLEM_3 = """다음 정적분을 계산하고 기하학적 의미를 설명해주세요.
∫₀² (x² + 2x) dx

1. 부정적분을 먼저 구하세요.
2. 정적분 값을 계산하세요.
3. 이 값의 기하학적 의미를 설명하세요."""

# TODO: math_graph로 테스트하세요
print("문제 3:", PROBLEM_3[:80], "...")

---
## §7 Phoenix Evaluation 실행

구현이 완료되면 평가를 실행하고 결과를 제출하세요.

In [ ]:
# 수학 풀이 평가 데이터셋 생성
import pandas as pd

math_eval_data = [
    {"input": PROBLEM_1, "expected": "극값", "type": "calculus"},
    {"input": PROBLEM_2, "expected": "x=1, y=3", "type": "matrix"},
    {"input": PROBLEM_3, "expected": "20/3", "type": "calculus"},
]
math_eval_df = pd.DataFrame(math_eval_data)

# TODO: 구현 완료 후 평가 실행
# results = []
# for _, row in math_eval_df.iterrows():
#     result = math_graph.invoke({...})
#     results.append({...})

# Phoenix에 experiment 기록
# → Experiments 탭 URL을 제출하세요!

print("TODO: 구현 완료 후 평가를 실행하고 Phoenix experiment URL을 제출하세요.")
print("http://localhost:6006 → Experiments 탭")

---
## 📋 제출 체크리스트

- [ ] `MathAgentState`에 `problem`, `solution_steps` 필드 추가
- [ ] `mode_router_3way` 프롬프트 완성 (math_solver 구분 기준 명확히)
- [ ] `math_solver_node` 구현 (Phoenix 프롬프트 Pull 포함)
- [ ] 전체 그래프 조립 및 컴파일
- [ ] 테스트 케이스 3개 실행
- [ ] Phoenix Evaluation 실행
- [ ] **Phoenix experiment URL 제출**

### 힌트
1. `math_solver_node`에서 Tool을 직접 호출할 필요는 없습니다. `llm_with_tools`를 사용하면 LLM이 알아서 판단합니다.
2. `solution_steps`에는 각 추론 단계를 문자열 리스트로 저장하세요.
3. Phoenix Trace를 보면 어떤 Tool이 호출됐는지 확인할 수 있습니다.
4. 잘 안 되면 `calculator_node`의 구현 방식을 참고하세요.